<a href="https://colab.research.google.com/github/fatahrahimi330/XST-Deepfake-Detection/blob/master/Prediction/Video_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# We use CNN + ViT + BiLSTM model to predict whether a video is deepfake or real.

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms
from facenet_pytorch import MTCNN
from CNN_ViT_BiLSTM import CNN_ViT_BiLSTM
import torch.nn as nn
import timm

## 1. Load the Trained Model

In [ ]:
class CNN_ViT_BiLSTM(nn.Module):
    def __init__(self, cnn_model='efficientnet_b0', vit_model='vit_base_patch16_224', lstm_hidden=256, lstm_layers=1):
        super(CNN_ViT_BiLSTM, self).__init__()

        # CNN backbone (pretrained)
        self.cnn = timm.create_model(cnn_model, pretrained=True)
        self.cnn.reset_classifier(0)  # remove classifier to get features
        cnn_feature_dim = self.cnn.num_features

        # Freeze CNN
        for param in self.cnn.parameters():
            param.requires_grad = False

        # ViT backbone (pretrained)
        self.vit = timm.create_model(vit_model, pretrained=True)
        self.vit.reset_classifier(0)
        vit_feature_dim = self.vit.num_features

         # Freeze ViT
        for param in self.vit.parameters():
            param.requires_grad = False

        # BiLSTM for temporal modeling
        self.lstm = nn.LSTM(
            input_size=cnn_feature_dim + vit_feature_dim, # Corrected input size
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True
        )


        # Fully connected layer for binary output
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden*2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        """
        x: shape (B, T, C, H, W)
        B = batch size, T = number of frames per video
        """
        B, T, C, H, W = x.shape
        x_flat = x.view(B*T, C, H, W) # Flatten B and T dimensions for CNN and ViT processing

        # CNN feature extraction from original images
        cnn_raw_feat = self.cnn.forward_features(x_flat)
        # Apply global pooling and flatten for CNN
        cnn_feat = self.cnn.global_pool(cnn_raw_feat).flatten(1)

        # ViT feature extraction from original images (parallel processing)
        vit_raw_feat = self.vit.forward_features(x_flat)
        # Extract CLS token for ViT
        vit_feat = vit_raw_feat[:, 0]

        # Concatenate features
        combined_feat = torch.cat((cnn_feat, vit_feat), dim=1) # Combine features

        # Reshape to sequence for LSTM
        seq_feat = combined_feat.view(B, T, -1)

        # BiLSTM
        lstm_out, _ = self.lstm(seq_feat)  # lstm_out: (B, T, 2*hidden)
        lstm_last = lstm_out[:, -1, :]     # take last frame output

        # FC layer
        out = self.fc(lstm_last)
        return out

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CNN_ViT_BiLSTM()  # same architecture as during training
model.load_state_dict(torch.load("/content/drive/MyDrive/copy_best_model.pth", map_location=device))
model.to(device)
model.eval()  # important for evaluation!

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# CRITICAL FIX: Unfreeze conv_head so gradients can flow for Grad-CAM
# ═════════════════════════════════════════════════════════════════════════════
for param in model.cnn.conv_head.parameters():
    param.requires_grad = True

In [ ]:
# ── Classes (must match ImageFolder alphabetical order) ───────────────────────
classes = ["fake", "real"]  # fake=0, real=1

In [ ]:
# ── MTCNN (CPU only) ──────────────────────────────────────────────────────────
mtcnn = MTCNN(keep_all=False, device="cpu")

In [ ]:
import torch
import torch.nn.functional as F
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms
from facenet_pytorch import MTCNN
from CNN_ViT_BiLSTM import CNN_ViT_BiLSTM
import matplotlib.pyplot as plt
import os

class GradCAM:
    """
    Grad-CAM for CNN_ViT_BiLSTM.

    Architecture flow:
      (B, T, C, H, W)
           ↓  flatten to (B*T, C, H, W)
       EfficientNet-B0 → conv_head → (B*T, 1280, H_feat, W_feat)
           ↓  global_pool + flatten → (B*T, 1280)
       concat ViT CLS → (B*T, 1280+768)
           ↓  reshape → (B, T, 2048)
       BiLSTM → FC → scalar logit

    We hook into conv_head to get spatial activation maps per frame.
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None

        self._fwd = target_layer.register_forward_hook(self._save_activation)
        self._bwd = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        # output: (B*T, 1280, H_feat, W_feat)
        self.activations = output.detach().clone()

    def _save_gradient(self, module, grad_input, grad_output):
        # grad_output[0]: (B*T, 1280, H_feat, W_feat)
        self.gradients = grad_output[0].detach().clone()

    def generate(self, input_tensor, target="fake"):
        """
        Args:
            input_tensor : (1, T, C, H, W) on device, requires_grad=True
            target       : "fake" or "real" — which class to explain

        Returns:
            cam_per_frame : list of T numpy arrays (H, W) in [0, 1]
            prob_fake     : float
            prob_real     : float
            pred_label    : str
        """
        self.model.zero_grad()

        # Forward pass
        logit = self.model(input_tensor)          # (1, 1)
        prob_real = torch.sigmoid(logit).item()
        prob_fake = 1.0 - prob_real
        pred_label = "real" if prob_real > 0.5 else "fake"

        # Backward pass
        # High logit → REAL, Low logit → FAKE
        # To explain FAKE: minimize logit (negate), to explain REAL: maximize
        if target == "fake":
            logit.backward(torch.ones_like(logit) * -1)  # gradient toward fake
        else:
            logit.backward(torch.ones_like(logit))        # gradient toward real

        # ── Compute CAM ──────────────────────────────────────────────────────
        grads = self.gradients    # (B*T, 1280, H_feat, W_feat)
        acts  = self.activations  # (B*T, 1280, H_feat, W_feat)

        # Global average pool gradients → importance weights per channel
        weights = grads.mean(dim=(2, 3), keepdim=True)  # (B*T, 1280, 1, 1)

        # Weighted combination of activation maps
        cam = (weights * acts).sum(dim=1)  # (B*T, H_feat, W_feat)
        cam = F.relu(cam)                  # only keep positive influence

        # Normalize per frame to [0, 1]
        cam_per_frame = []
        T = cam.shape[0]
        for t in range(T):
            c = cam[t].cpu().numpy()
            c_min, c_max = c.min(), c.max()
            c = (c - c_min) / (c_max - c_min + 1e-8)
            cam_per_frame.append(c)

        return cam_per_frame, prob_fake, prob_real, pred_label

    def remove_hooks(self):
        self._fwd.remove()
        self._bwd.remove()

In [ ]:
# ── Exact same helper as training ─────────────────────────────────────────────
def expand_box(x1, y1, x2, y2, w, h, margin_ratio=0.25):
    bw = x2 - x1
    bh = y2 - y1
    mx = int(bw * margin_ratio)
    my = int(bh * margin_ratio)
    x1 = max(0, x1 - mx)
    y1 = max(0, y1 - my)
    x2 = min(w, x2 + mx)
    y2 = min(h, y2 + my)
    return x1, y1, x2, y2

## 2. Preprocess video into frames

In [ ]:
def overlay_heatmap(frame_rgb, cam, alpha=0.45):
    """Overlay Grad-CAM heatmap on a frame."""
    h, w = frame_rgb.shape[:2]
    cam_resized = cv2.resize(cam, (w, h))
    heatmap_bgr = cv2.applyColorMap(
        (cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET
    )
    heatmap_rgb = cv2.cvtColor(heatmap_bgr, cv2.COLOR_BGR2RGB)
    blended = (alpha * heatmap_rgb + (1 - alpha) * frame_rgb).astype(np.uint8)
    return blended, cam_resized


def save_gradcam_grid(frames, cam_list, prob_fake, prob_real, pred_label,
                      output_path="gradcam_output.png"):
    """
    Two-row grid:
      Row 1 — original face frames
      Row 2 — Grad-CAM overlays with per-frame activation score
    """
    T = len(frames)
    fig, axes = plt.subplots(2, T, figsize=(T * 2.5, 6))
    fig.patch.set_facecolor("#111111")

    color = "#ff4444" if pred_label == "fake" else "#44dd88"
    fig.suptitle(
        f"Prediction: {pred_label.upper()}    "
        f"Fake {prob_fake:.1%}  |  Real {prob_real:.1%}",
        color=color, fontsize=13, fontweight="bold", y=1.02
    )

    for t in range(T):
        overlay, cam_up = overlay_heatmap(frames[t], cam_list[t])
        act_score = cam_up.mean()

        # Original frame
        ax = axes[0, t]
        ax.imshow(frames[t])
        ax.set_title(f"F{t+1}", color="white", fontsize=7)
        ax.axis("off")

        # Grad-CAM overlay
        ax = axes[1, t]
        ax.imshow(overlay)
        ax.set_title(
            f"{act_score:.2f}",
            color="tomato" if act_score > 0.35 else "gray",
            fontsize=7
        )
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"[✓] Saved grid → {output_path}")


def save_top_suspicious_frames(frames, cam_list, prob_fake, pred_label,
                                top_k=3, out_dir="gradcam_top"):
    """Save top-k most activated frames side-by-side (original | heatmap)."""
    os.makedirs(out_dir, exist_ok=True)
    scores = [cam.mean() for cam in cam_list]
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

    for rank, idx in enumerate(top_idx):
        overlay, _ = overlay_heatmap(frames[idx], cam_list[idx])
        color = "#ff4444" if pred_label == "fake" else "#44dd88"

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3.5))
        fig.patch.set_facecolor("#111111")
        fig.suptitle(
            f"{pred_label.upper()}  |  Frame {idx+1}  |  "
            f"Activation: {scores[idx]:.3f}  |  Fake prob: {prob_fake:.1%}",
            color=color, fontsize=10
        )
        ax1.imshow(frames[idx]); ax1.set_title("Original", color="white", fontsize=9); ax1.axis("off")
        ax2.imshow(overlay);     ax2.set_title("Grad-CAM", color="white", fontsize=9); ax2.axis("off")
        plt.tight_layout()
        out_path = os.path.join(out_dir, f"rank{rank+1}_frame{idx+1}.png")
        plt.savefig(out_path, dpi=150, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        plt.close()
        print(f"[✓] Saved top frame {rank+1} → {out_path}")

In [ ]:

def expand_box(x1, y1, x2, y2, w, h, margin_ratio=0.25):
    bw, bh = x2 - x1, y2 - y1
    mx, my = int(bw * margin_ratio), int(bh * margin_ratio)
    return max(0, x1-mx), max(0, y1-my), min(w, x2+mx), min(h, y2+my)


def video_to_frames(video_path, num_frames=16, img_size=224,
                    frame_skip=10, margin_ratio=0.25):
    cap = cv2.VideoCapture(video_path)
    frames, frame_id = [], 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_id % frame_skip == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            boxes, _ = mtcnn.detect(rgb)
            if boxes is not None and len(boxes) > 0:
                x1, y1, x2, y2 = map(int, boxes[0])
                h, w, _ = rgb.shape
                x1, y1, x2, y2 = expand_box(x1, y1, x2, y2, w, h, margin_ratio)
                face = rgb[y1:y2, x1:x2]
                if face.size > 0:
                    frames.append(cv2.resize(face, (img_size, img_size)))
        frame_id += 1
        if len(frames) == num_frames:
            break
    cap.release()
    if not frames:
        raise ValueError("No faces detected!")
    while len(frames) < num_frames:
        frames.append(frames[-1])
    return np.stack(frames)

## 3. Apply the same transformations as training

In [ ]:
# ── Transforms (matches val_test_transform in training) ──────────────────────
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [ ]:
# ── Preprocess frames ─────────────────────────────────────────────────────────
def preprocess_frames(frames, transform):
    tensors = torch.stack([transform(Image.fromarray(f)) for f in frames])
    return tensors.unsqueeze(0)  # (1, T, C, H, W)

## 4. Predict

In [ ]:
video_path = "01_02__outside_talking_still_laughing__YVGY8LOK.mp4"

# 1. Extract raw face frames (keep uint8 for visualization)
print("Extracting frames...")
frames = video_to_frames(video_path, num_frames=16)  # (T, H, W, 3)

# 2. Preprocess for model
x = preprocess_frames(frames, val_transform).to(device)

# 3. Setup Grad-CAM on EfficientNet's last conv layer
grad_cam = GradCAM(model, target_layer=model.cnn.conv_head)

# 4. Generate explanation (explain toward "fake" class)
print("Running Grad-CAM...")
cam_list, prob_fake, prob_real, pred_label = grad_cam.generate(x, target="fake")
grad_cam.remove_hooks()

# 5. Results
print(f"\nProbability of being Fake : {prob_fake:.4f}")
print(f"Probability of being Real : {prob_real:.4f}")
print(f"Prediction                : {pred_label.upper()}")

# 6. Save visualizations
save_gradcam_grid(frames, cam_list, prob_fake, prob_real, pred_label)
save_top_suspicious_frames(frames, cam_list, prob_fake, pred_label, top_k=3)